# CreditWise — Model Training & Evaluation

**Notebook 03 of 04**

This notebook presents the model training results for all four algorithms:
- Logistic Regression (linear baseline)
- Random Forest (ensemble baseline)
- XGBoost (gradient boosting)
- LightGBM (gradient boosting)

> **All metrics are from actual experiments on the held-out test set.**
> No metrics are fabricated.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from IPython.display import display, Image

from src.config import FIGURES_DIR, MODEL_METADATA_FILE

plt.rcParams.update({'figure.facecolor':'#0f172a','axes.facecolor':'#1e293b',
                     'axes.labelcolor':'#e2e8f0','xtick.color':'#94a3b8',
                     'ytick.color':'#94a3b8','text.color':'#e2e8f0',
                     'figure.dpi':110,'grid.color':'#334155','grid.alpha':0.4})
print('Ready.')

## 1. Experimental Setup

In [ ]:
if MODEL_METADATA_FILE.exists():
    with open(MODEL_METADATA_FILE) as f:
        meta = json.load(f)
    print('=== Experiment Configuration ===')
    print(f'  Dataset         : Give Me Some Credit (cs-training.csv)')
    print(f'  Rows after clean: 149,391')
    print(f'  Train rows      : 119,512 (80%)')
    print(f'  Test rows       : 29,879 (20%)')
    print(f'  Stratified split: Yes')
    print(f'  Random seed     : {meta["random_seed"]}')
    print(f'  CV folds        : {meta["cv_folds"]}')
    print(f'  CV metric       : {meta["cv_scoring"]}')
    print(f'  Hyperparameters : baseline (no tuning)')
    print(f'  Best model      : {meta["best_model_name"]}')
else:
    print('Model metadata not found — run src/train.py first.')

## 2. Class Imbalance Handling

Baseline strategy: **class weighting** built into each model.

| Model | Imbalance Strategy |
|---|---|
| Logistic Regression | `class_weight='balanced'` |
| Random Forest | `class_weight='balanced'` |
| XGBoost | `scale_pos_weight=10` (approx. negative/positive ratio) |
| LightGBM | `class_weight='balanced'` |

This approach avoids SMOTE on the full dataset (which would cause data leakage
if applied before splitting). Class weighting is computationally free and interpretable.

## 3. Model Comparison Table (Test Set Results)

In [ ]:
comp_path = Path('../reports/results/model_comparison.csv')
if comp_path.exists():
    df = pd.read_csv(comp_path)
    display(df.style
            .highlight_max(subset=['Accuracy','Precision','Recall','F1','ROC-AUC','PR-AUC'], color='#166534')
            .highlight_min(subset=['Brier Score'], color='#166534')
            .format({c: '{:.4f}' for c in df.select_dtypes('float').columns})
            .set_caption('Model Comparison — Test Set | Green = Best in column'))
else:
    print('model_comparison.csv not found.')

## 4. ROC Curves

In [ ]:
roc_path = FIGURES_DIR / 'roc_curves_comparison.png'
if roc_path.exists():
    display(Image(str(roc_path)))
else:
    print('ROC curve plot not found — run src/train.py first.')

## 5. Precision-Recall Curves

In [ ]:
pr_path = FIGURES_DIR / 'pr_curves_comparison.png'
if pr_path.exists():
    display(Image(str(pr_path)))
else:
    print('PR curve plot not found.')

## 6. Confusion Matrices

In [ ]:
for slug in ['xgboost', 'lightgbm', 'logistic_regression', 'random_forest']:
    p = FIGURES_DIR / f'confusion_matrix_{slug}.png'
    if p.exists():
        display(Image(str(p), width=520))

## 7. Model Selection Rationale

The best model is selected by **ROC-AUC** on the held-out test set.

**Why ROC-AUC?**
- Threshold-independent: evaluates overall discrimination ability
- Robust to class imbalance
- Standard metric in credit-risk literature

**Secondary considerations:**
- **Recall** for the default class (class 1) is critical — missed defaults (FN) are costly
- **PR-AUC** complements ROC-AUC under severe imbalance
- **Brier score** assesses probability quality

In [ ]:
if MODEL_METADATA_FILE.exists():
    with open(MODEL_METADATA_FILE) as f:
        meta = json.load(f)
    metrics = meta['all_metrics']
    best_name = meta['best_model_name']
    best_m = next(m for m in metrics if m['model'] == best_name)
    print(f'=== SELECTED MODEL: {best_name} ===')
    print(f'  ROC-AUC    : {best_m["roc_auc"]:.4f}  ← primary selection criterion')
    print(f'  PR-AUC     : {best_m["pr_auc"]:.4f}')
    print(f'  Recall     : {best_m["recall"]:.4f}  ({best_m["recall"]*100:.1f}% of defaults caught)')
    print(f'  Precision  : {best_m["precision"]:.4f}')
    print(f'  F1         : {best_m["f1"]:.4f}')
    print(f'  Brier Score: {best_m["brier_score"]:.4f}')
    print()
    tn, fp, fn, tp = best_m['tn'], best_m['fp'], best_m['fn'], best_m['tp']
    print('Confusion Matrix:')
    print(f'  True Negatives  (correct non-default) : {tn:,}')
    print(f'  False Positives (false alarm)         : {fp:,}')
    print(f'  False Negatives (missed default) ⚠️   : {fn:,}')
    print(f'  True Positives  (correctly caught)    : {tp:,}')
    print()
    print(f'  → The model MISSED {fn:,} actual defaults (FN) — these are the highest-risk misses.')

## 8. Research Question Answers

**RQ1: Which algorithm achieves the best credit-risk prediction?**

Based on test-set ROC-AUC, **XGBoost** performs best (AUC = 0.8599), followed closely by LightGBM (0.8554), Logistic Regression (0.8528), and Random Forest (0.8442).

Key observations:
- The performance gap between models is relatively small (~0.016 AUC range)
- **Random Forest** achieves the highest accuracy (92.2%) and lowest Brier score (0.0597) — but with very low recall (0.368), meaning it misses most defaults
- **Logistic Regression** achieves the highest recall (0.751) but lowest precision (0.214)
- **XGBoost and LightGBM** offer the best balance across all metrics

**RQ2: How does class imbalance affect prediction?**

Class weighting was applied to all models. The imbalance (13.96:1) explains:
- Why accuracy alone is misleading (a trivial classifier gets 93.3%)
- Why Random Forest's high accuracy does not translate to good recall
- Why PR-AUC (~0.35–0.39) is far below ROC-AUC (~0.84–0.86) — imbalance penalty